In [ ]:
from pathlib import Path

MODEL='Version4.1'

DATASET_PATH=Path('preprocessed_data')

DATASET_PATH.mkdir(parents=True, exist_ok=True)

EVALUATE = True

DATASETS_TO_EXCLUDE = [] #Datasets to exclude from evaluation

DATASET_TO_CHECK = ['tr', 'tr_vertical'] #Datasets to show images and example predictions from

In [ ]:
import json
def get_word_files(dataset_path, isTr=False):
    with open(str(dataset_path / 'labels.json'), 'r') as fp:
        data_dict = json.load(fp)

    if isTr:
        word_files = list(data_dict['labels'].items())
        words_files = list()
        for file in word_files:
             words_files.append((str(dataset_path / 'images' / file[0]) + '.jpg', file[1]))
    else:
        words_files = list(data_dict.items())
    print(words_files[0])
    print(f"{len(words_files)} words")
    return(words_files)

In [ ]:
kmnist_words = get_word_files(DATASET_PATH / 'kmnist')
k49_words = get_word_files(DATASET_PATH / 'K49')
kanjivg_words = get_word_files(DATASET_PATH / 'kanjivg')
kkanji2_words = get_word_files(DATASET_PATH / 'kkanji2')
tr_words = get_word_files(DATASET_PATH / 'text_renderer', isTr=True)
tr_print_words = get_word_files(DATASET_PATH / 'text_renderer_print', isTr=True)
tr_vertical_words = get_word_files(DATASET_PATH / 'text_renderer_vertical', isTr=True)

words_list = [(kmnist_words, 'kmnist'), (k49_words, 'k49'), (kanjivg_words, 'kanjivg'), (kkanji2_words, 'kkanji2'), (tr_words, 'tr'), (tr_print_words, 'tr_print'), (tr_vertical_words, 'tr_vertical')]

In [ ]:
from typing import Tuple
import tqdm
import torch
from dtrocr.config import DTrOCRConfig
from torch.utils.data import DataLoader
import torch
torch.set_float32_matmul_precision('high')
from dtrocr.model import DTrOCRLMHeadModel

def evaluate_model(model: torch.nn.Module, dataloader: DataLoader) -> Tuple[float, float]:
    # set model to evaluation mode
    model.eval()
    
    losses, accuracies = [], []
    with torch.no_grad():
        for inputs in tqdm.tqdm(dataloader, total=len(dataloader), desc=f'Evaluating test set'):
            inputs = send_inputs_to_device(inputs, device=0)
            outputs = model(**inputs)
            
            losses.append(outputs.loss.item())
            accuracies.append(outputs.accuracy.item())
    
    loss = sum(losses) / len(losses)
    accuracy = sum(accuracies) / len(accuracies)
    
    # set model back to training mode
    model.train()
    
    return loss, accuracy

def send_inputs_to_device(dictionary, device):
    return {key: value.to(device=device) if isinstance(value, torch.Tensor) else value for key, value in dictionary.items()}



model = DTrOCRLMHeadModel(DTrOCRConfig())
model = torch.compile(model)
model.load_state_dict(torch.load(f'../models/{MODEL}.pt'))
model.to(device=0)



In [ ]:
from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig
from pathlib import Path
from dataclasses import dataclass
from PIL import Image
from torch.utils.data import Dataset
  
@dataclass
class Word:
    file_path: Path
    transcription: str
    

def get_word(word_file):
    words = []
    words.append(
        Word(
            file_path=word_file[0],
            transcription=word_file[1]
        )
    )
    return words

    
class IAMDataset(Dataset):
    def __init__(self, words: list[Word], config: DTrOCRConfig):
        super(IAMDataset, self).__init__()
        self.words = words
        self.processor = DTrOCRProcessor(config, add_eos_token=True, add_bos_token=True)
        
    def __len__(self):
        return len(self.words)
    
    def __getitem__(self, item):
        inputs = self.processor(
            images=Image.open(self.words[item].file_path).convert('RGB'),
            texts=self.words[item].transcription,
            padding='max_length',
            return_tensors="pt",
            return_labels=True,
        )
        return {
            'pixel_values': inputs.pixel_values[0],
            'input_ids': inputs.input_ids[0],
            'input_attention_mask': inputs.input_attention_mask[0],
            'label_attention_mask': inputs.label_attention_mask[0],
            'labels': inputs.labels[0]
        }

config = DTrOCRConfig(
    # attn_implementation='flash_attention_2'
)




In [ ]:
import tqdm
import multiprocessing as mp

train_word_records = {}
for pair in words_list:
    if pair[1] in DATASETS_TO_EXCLUDE : continue
    with mp.Pool(processes=mp.cpu_count()) as pool:
        word_list = list(
            tqdm.tqdm(
                pool.imap(get_word, pair[0]), 
                total=len(pair[0]),
                desc='Building dataset'
            )
        )
    words = [word for words_array in word_list for word in words_array]
    train_word_records[pair[1]] = words
    print(words[0])
    if EVALUATE:
        data = IAMDataset(words=words, config=config)
        dataloader = DataLoader(data, batch_size=32, shuffle=True, num_workers=mp.cpu_count())
        loss, accuracy = evaluate_model(model, dataloader)
            
        print(f"{pair[1]} loss: {loss}, accuracy: {accuracy}")

In [ ]:
from dtrocr.processor import DTrOCRProcessor
from dtrocr.config import DTrOCRConfig

model.eval()
model.to('cpu')
test_processor = DTrOCRProcessor(DTrOCRConfig())

In [ ]:
import matplotlib

matplotlib.rc('font', family='TakaoPGothic')

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

print(train_word_records[DATASET_TO_CHECK[0]])

for ds in DATASET_TO_CHECK:
    for test_word_record in train_word_records[ds[0]][:10]:
        image_file = test_word_record.file_path
        image = Image.open(image_file).convert('RGB')
        
        inputs = test_processor(
            images=image, 
            texts='',
            return_tensors='pt'
        )
        
        model_output = model.generate(
            inputs, 
            test_processor,
            num_beams=1
        )
        
        predicted_text = test_processor.tokeniser.decode(model_output[0], skip_special_tokens=False)
        
        print(test_word_record.transcription, ' ', predicted_text, ' ', inputs.input_ids)
        plt.figure(figsize=(10, 5))
        plt.title(predicted_text, fontsize=24)
        plt.imshow(np.array(image, dtype=np.uint8))
        plt.xticks([]), plt.yticks([])
        plt.show()

In [ ]:
tokeniser = test_processor.tokeniser
print(tokeniser.pad_token)
print(tokeniser.eos_token)
print(tokeniser.bos_token)
print(tokeniser.model_max_length)
print(tokeniser.sep_token_id)

print(tokeniser('<s>[SEP]'))